# Reservoir Water Level Analysis — DAHITI Satellite Data

**Purpose:** Loads and plots satellite-derived water level time series from
DAHITI for multiple reservoirs (Talsperren) in the Ziegenrück region, and
compares them against simulated or reference levels.

**What it does:**
- Reads multiple sheets from the DAHITI Excel file
  (`Alter_Teich`, `Mittel_Teich`, `Teich_3`, `Fürstenteich`, etc.)
- Parses and aligns datetime indices
- Generates time-series plots of observed water levels per reservoir

**Input:** `DAHITI Water Level data.xlsx`  
**Output:** Water level time-series plots

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import matplotlib.ticker as mticker

In [ ]:
file_path = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\DAHITI Water Level data.xlsx"

# Sheet names to plot (order matters)
sheets = [
    "Alter_Teich",
    "Mittel_Teich",
    "Teich_3",
    "Fürstenteich",
    "Mahlteich",
    "Mittelteich",
    "Moosteich",
    "Genscherodteich"
]

# Read all sheets into a dictionary
dfs = {}
for sheet in sheets:
    df = pd.read_excel(file_path, sheet_name=sheet)
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
    dfs[sheet] = df

# Create subplots
fig, axes = plt.subplots(len(sheets), 1, figsize=(14, 3.5 * len(sheets)), sharex=True)

# Colors & markers (optional but clearer)
colors = ["blue", "green", "cyan", "red", "orange", "purple", "brown", "teal"]
markers = ["o", "s", "o", "^", "D", "v", "P", "X"]

# Plot each Teich
for i, (sheet, df) in enumerate(dfs.items()):
    axes[i].plot(
        df["Date"],
        df["WL"],
        lw=1.5,
        marker=markers[i],
        label=sheet,
        color=colors[i]
    )

    axes[i].set_ylabel("Water Level [m]")
    axes[i].set_title(f"{sheet} Water Level")
    axes[i].grid(True, alpha=0.5)
    axes[i].legend(loc="upper left")

    # Y-axis ticks every 0.5 m
    wl_min = df["WL"].min()
    wl_max = df["WL"].max()
    axes[i].set_yticks(
        np.arange(
            np.floor(wl_min * 2) / 2,
            np.ceil(wl_max * 2) / 2 + 0.5,
            0.5
        )
    )

# Shared X-axis formatting
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d.%m.%Y"))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
axes[-1].set_xlabel("Date")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# =============================================================================
# FILE PATHS
# =============================================================================
DAHITI_FILE = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\DAHITI Water Level data.xlsx"

# =============================================================================
# CONFIGURATION
# =============================================================================
SHEETS = [
    "Alter_Teich",
    "Mittel_Teich",
    "Teich_3",
    "Fürstenteich",
    "Mahlteich",
    "Mittelteich",
    "Moosteich",
    "Genscherodteich",
]

PANEL_LABELS = ["a", "b", "c", "d", "e", "f", "g", "h"]
DISPLAY_NAMES = [
    "Alter Teich",
    "Mittel Teich",
    "NSG Dreba-Plothener Teichgebiet",
    "Fürstenteich",
    "Mahlteich",
    "Mittelteich",
    "Moosteich",
    "Genscherodteich",
]

DAHITI_COLOR = "#1f77b4"

# =============================================================================
# HELPERS
# =============================================================================

def fmt_xaxis(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator())
    ax.tick_params(axis="x", which="major", labelsize=5.5)
    ax.tick_params(axis="x", which="minor", length=2, width=0.5, color="0.5")
    # Light vertical grid lines at each year (major) and subtle at months (minor)
    ax.grid(True, which="major", axis="x", linestyle="--", linewidth=0.4, alpha=0.5, color="0.4")
    ax.grid(True, which="minor", axis="x", linestyle=":",  linewidth=0.3, alpha=0.3, color="0.5")


def style_ax(ax):
    ax.tick_params(axis="y", labelsize=4.5)
    ax.tick_params(axis="both", which="both", direction="out")
    ax.grid(False, axis="y")   # y-grid off; x-grid set in fmt_xaxis
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.6)
    ax.spines["bottom"].set_linewidth(0.6)


# =============================================================================
# LOAD DAHITI DATA
# =============================================================================
dfs = {}
for sheet in SHEETS:
    df = pd.read_excel(DAHITI_FILE, sheet_name=sheet)
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
    df = df.dropna(subset=["Date", "WL"]).sort_values("Date")
    dfs[sheet] = df

# =============================================================================
# FIGURE
# =============================================================================
N = len(SHEETS)
fig, axes = plt.subplots(
    N, 1,
    figsize=(6.30, 1.15 * N),
    sharex=False,
    gridspec_kw={"hspace": 0.45}
)

for i, (sheet, label, name) in enumerate(zip(SHEETS, PANEL_LABELS, DISPLAY_NAMES)):
    ax = axes[i]
    df = dfs[sheet]

    wl_mean = df["WL"].mean()
    ax.axhline(wl_mean, color="0.5", linewidth=0.6, linestyle="--", zorder=1)

    ax.plot(df["Date"], df["WL"], color=DAHITI_COLOR, linewidth=0.7, zorder=2)

    ax.fill_between(df["Date"], df["WL"], wl_mean,
                    where=(df["WL"] >= wl_mean),
                    alpha=0.22, color=DAHITI_COLOR, zorder=1)
    ax.fill_between(df["Date"], df["WL"], wl_mean,
                    where=(df["WL"] < wl_mean),
                    alpha=0.15, color="red", zorder=1)

    wl_min, wl_max = df["WL"].min(), df["WL"].max()
    ax.set_yticks(np.arange(
        np.floor(wl_min * 2) / 2,
        np.ceil(wl_max  * 2) / 2 + 0.5,
        0.5
    ))

    ax.set_ylabel("Water Level [m]", fontsize=4.5)
    ax.set_title(f"{label}) {name}", fontsize=6.5, fontweight="bold", pad=3, loc="left")
    ax.set_xlim(df["Date"].min(), df["Date"].max())

    fmt_xaxis(ax)
    style_ax(ax)

    if i == N - 1:
        ax.set_xlabel("Year/Months", fontsize=6)

# =============================================================================
# GROUP ANNOTATIONS  (right-side brackets)
# =============================================================================
# DAHITI group: panels 0-1 (Alter Teich, Mittel Teich)
# TLG group:    panels 2-6 (Fürstenteich … Genscherodteich)
fig.canvas.draw()

def add_group_bracket(fig, axes_group, label, color):
    """Draw a right-side curly bracket spanning the given axes with a rotated label."""
    # Get top of first ax and bottom of last ax in figure fraction
    top_ax  = axes_group[0]
    bot_ax  = axes_group[-1]
    fig_h   = fig.get_size_inches()[1] * fig.dpi

    top_bb  = top_ax.get_position()
    bot_bb  = bot_ax.get_position()

    y_top = top_bb.y1
    y_bot = bot_bb.y0
    y_mid = (y_top + y_bot) / 2
    x_right = 1.01   # just outside right edge in figure fraction

    # Vertical line
    fig.add_artist(plt.Line2D(
        [x_right, x_right], [y_bot, y_top],
        transform=fig.transFigure,
        color=color, linewidth=1.2, clip_on=False
    ))
    # Small horizontal end caps
    cap = 0.005
    for y in [y_top, y_bot]:
        fig.add_artist(plt.Line2D(
            [x_right - cap, x_right], [y, y],
            transform=fig.transFigure,
            color=color, linewidth=1.2, clip_on=False
        ))
    # Rotated label
    fig.text(
        x_right + 0.015, y_mid, label,
        ha="left", va="center", rotation=270,
        fontsize=5.5, color=color, fontweight="bold",
        transform=fig.transFigure, clip_on=False
    )

add_group_bracket(fig, axes[0:3],  "DAHITI",                        "#1f77b4")
add_group_bracket(fig, axes[3:],   "Thüringer Landesgesellschaft",  "#d62728")

# =============================================================================
# SAVE
# =============================================================================
fig.tight_layout()
fig.subplots_adjust(hspace=0.45)
plt.savefig("water_level_plot.png", dpi=300, bbox_inches="tight")
print("Saved → water_level_plot.png")
plt.show()

if __name__ == "__main__":
    pass